<a id="title"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:150%; text-align:center; border-radius:20px 20px;">**Reference Area Estimation**</p>

## Method Overview

This notebook implements the **next sub-task of Block 2 — Geometric Stenosis Quantification**: for each branch of `Normal_1`, it computes the cumulative **geodesic distance** along the centerline and the local **reference area** used for stenosis quantification.

### Inputs

- Branch dataframes from: `results/block2_results/area/samples/Normal_1/branches/dataframes`
- Required columns per branch: `Px`, `Py`, `Pz`, `Area`

### Quantities computed

1. **Geodesic distance** `gd` (mm):
   \[
   gd_i = \sum_{k=1}^{i} \sqrt{(\Delta x_k)^2 + (\Delta y_k)^2 + (\Delta z_k)^2}
   \]
2. **Sliding-window reference areas** with `window_size = 10` mm:
   - proximal target: `gd_i - 10`
   - distal target: `gd_i + 10`
   - nearest points in `gd` are used to map `Area_prox` and `Area_dist`
3. **Reference area**:
   \[
   A_{ref} = \frac{Area_{prox} + Area_{dist}}{2}
   \]

If either proximal or distal target falls outside branch limits, the corresponding area is `NaN`, so `A_ref` is also `NaN`. This naturally excludes the first and last 10 mm of each branch.

<a id="step1"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">1. Imports & Configuration</p>

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PATIENT_ID = "Normal_1"
window_size = 10.0  # mm

PROJECT_ROOT = Path.cwd().resolve().parents[1]
BRANCH_DF_DIR = PROJECT_ROOT / "results" / "block2_results" / "area" / "samples" / PATIENT_ID / "branches" / "dataframes"
OUTPUT_DIR = PROJECT_ROOT  # experimental output at repository root

print(f"Patient ID     : {PATIENT_ID}")
print(f"Branch DF dir  : {BRANCH_DF_DIR} (exists={BRANCH_DF_DIR.exists()})")
print(f"Window size    : {window_size} mm")
print(f"Output dir     : {OUTPUT_DIR}")

Patient ID     : Normal_1
Branch DF dir  : C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\results\block2_results\area\samples\Normal_1\branches\dataframes (exists=True)
Window size    : 10.0 mm
Output dir     : C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository


<a id="step2"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">2. Data Loading & Branch Discovery</p>

This section auto-discovers branch files (`.xlsx` preferred, `.csv` fallback), validates required columns, and loads each branch dataframe into a dictionary keyed by branch name.

In [3]:
def get_branch_name(file_path: Path) -> str:
    name = file_path.stem
    for suffix in ["_area", "_areas", "_sectional_area"]:
        if name.endswith(suffix):
            name = name[: -len(suffix)]
    return name


def discover_branch_files(branch_dir: Path):
    if not branch_dir.exists():
        raise FileNotFoundError(
            f"Branch directory does not exist: {branch_dir}\n"
            "Update BRANCH_DF_DIR to your actual location."
        )

    xlsx_files = sorted(branch_dir.glob("*.xlsx"))
    csv_files = sorted(branch_dir.glob("*.csv"))

    if xlsx_files:
        return xlsx_files, "xlsx"
    if csv_files:
        return csv_files, "csv"

    raise FileNotFoundError(
        f"No .xlsx or .csv branch files found in: {branch_dir}"
    )


def load_branch_dataframe(file_path: Path) -> pd.DataFrame:
    if file_path.suffix.lower() == ".xlsx":
        df = pd.read_excel(file_path)
    elif file_path.suffix.lower() == ".csv":
        df = pd.read_csv(file_path)
    else:
        raise ValueError(f"Unsupported extension: {file_path.suffix}")

    required_cols = {"Px", "Py", "Pz", "Area"}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"{file_path.name} is missing columns: {sorted(missing_cols)}")

    return df


branch_files, source_format = discover_branch_files(BRANCH_DF_DIR)
branch_data = {}

for file_path in branch_files:
    branch_name = get_branch_name(file_path)
    branch_data[branch_name] = load_branch_dataframe(file_path)

print(f"Detected {len(branch_data)} branches from {source_format.upper()} files:")
for branch_name, branch_df in branch_data.items():
    print(f"- {branch_name}: {len(branch_df)} points")

Detected 11 branches from XLSX files:
- dataset_LCA_B01_Normal_1: 822 points
- dataset_LCA_B02_Normal_1: 425 points
- dataset_LCA_B03_Normal_1: 583 points
- dataset_LCA_B04_Normal_1: 575 points
- dataset_LCA_B05_Normal_1: 335 points
- dataset_LCA_B06_Normal_1: 316 points
- dataset_LCA_B07_Normal_1: 243 points
- dataset_RCA_B01_Normal_1: 1065 points
- dataset_RCA_B02_Normal_1: 1362 points
- dataset_RCA_B03_Normal_1: 1031 points
- dataset_RCA_B04_Normal_1: 530 points


<a id="step3"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">3. Geodesic Distance & Sliding-Window Reference Area</p>

For each branch dataframe:

- compute cumulative `gd` from ordered points (`Px`, `Py`, `Pz`),
- locate nearest points to `gd ± 10 mm`,
- map `Area_prox` and `Area_dist`,
- compute `A_ref` as their mean with standard NaN propagation.

In [4]:
def nearest_index_within_bounds(gd_values: np.ndarray, target_values: np.ndarray) -> np.ndarray:
    """
    Return nearest indices in gd_values for each target in target_values.
    Targets outside gd range are marked with -1.
    """
    idx = np.searchsorted(gd_values, target_values)
    idx = np.clip(idx, 0, len(gd_values) - 1)

    prev_idx = np.clip(idx - 1, 0, len(gd_values) - 1)

    dist_prev = np.abs(gd_values[prev_idx] - target_values)
    dist_curr = np.abs(gd_values[idx] - target_values)

    nearest_idx = np.where(dist_prev <= dist_curr, prev_idx, idx)

    out_of_bounds = (target_values < gd_values[0]) | (target_values > gd_values[-1])
    nearest_idx[out_of_bounds] = -1

    return nearest_idx


def compute_reference_columns(df_branch: pd.DataFrame, window_mm: float) -> pd.DataFrame:
    df_out = df_branch.copy()

    xyz = df_out[["Px", "Py", "Pz"]].to_numpy(dtype=float)
    delta_xyz = np.diff(xyz, axis=0)
    step_dist = np.linalg.norm(delta_xyz, axis=1)
    step_dist = np.insert(step_dist, 0, 0.0)

    df_out["gd"] = np.cumsum(step_dist)

    gd_vals = df_out["gd"].to_numpy(dtype=float)
    area_vals = df_out["Area"].to_numpy(dtype=float)

    prox_targets = gd_vals - window_mm
    dist_targets = gd_vals + window_mm

    prox_idx = nearest_index_within_bounds(gd_vals, prox_targets)
    dist_idx = nearest_index_within_bounds(gd_vals, dist_targets)

    area_prox = np.full(len(df_out), np.nan, dtype=float)
    area_dist = np.full(len(df_out), np.nan, dtype=float)

    prox_valid = prox_idx >= 0
    dist_valid = dist_idx >= 0

    area_prox[prox_valid] = area_vals[prox_idx[prox_valid]]
    area_dist[dist_valid] = area_vals[dist_idx[dist_valid]]

    df_out["Area_prox"] = area_prox
    df_out["Area_dist"] = area_dist
    df_out["A_ref"] = (df_out["Area_prox"] + df_out["Area_dist"]) / 2.0

    return df_out


processed_branch_data = {}

for branch_name, branch_df in branch_data.items():
    processed_branch_data[branch_name] = compute_reference_columns(
        branch_df,
        window_mm=window_size,
    )

first_branch_name = next(iter(processed_branch_data))
first_branch_df = processed_branch_data[first_branch_name]

valid_mask = first_branch_df["A_ref"].notna()
valid_pct = 100.0 * valid_mask.mean()
nan_pct = 100.0 - valid_pct

print(f"Verification branch: {first_branch_name}")
print(f"A_ref valid points: {valid_pct:.2f}%")
print(f"A_ref NaN points  : {nan_pct:.2f}%")

summary_df = pd.DataFrame(
    [
        {
            "branch": branch_name,
            "n_points": len(df_branch),
            "n_valid_A_ref": int(df_branch["A_ref"].notna().sum()),
            "n_nan_A_ref": int(df_branch["A_ref"].isna().sum()),
        }
        for branch_name, df_branch in processed_branch_data.items()
    ]
)

summary_df

Verification branch: dataset_LCA_B01_Normal_1
A_ref valid points: 83.45%
A_ref NaN points  : 16.55%


,branch,n_points,n_valid_A_ref,n_nan_A_ref
0,dataset_LCA_B01_Normal_1,822,686,136
1,dataset_LCA_B02_Normal_1,425,304,121
2,dataset_LCA_B03_Normal_1,583,466,117
3,dataset_LCA_B04_Normal_1,575,455,120
4,dataset_LCA_B05_Normal_1,335,211,124
5,dataset_LCA_B06_Normal_1,316,200,116
6,dataset_LCA_B07_Normal_1,243,125,118
7,dataset_RCA_B01_Normal_1,1065,936,129
8,dataset_RCA_B02_Normal_1,1362,1239,123
9,dataset_RCA_B03_Normal_1,1031,898,133


<a id="step4"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">4. Export Enriched Branch DataFrames</p>

Each processed branch is exported to the repository root as an experimental `.xlsx` file containing the new columns:

- `gd`
- `Area_prox`
- `Area_dist`
- `A_ref`

In [5]:
exported_files = []

for branch_name, df_branch in processed_branch_data.items():
    output_name = f"{PATIENT_ID}_{branch_name}_reference_values.xlsx"
    output_path = OUTPUT_DIR / output_name
    df_branch.to_excel(output_path, index=False)
    exported_files.append(output_path)

print("Export completed:")
for output_path in exported_files:
    print(f"- {output_path}")

preview_branch_name = first_branch_name
print(f"\nPreview of branch: {preview_branch_name}")
processed_branch_data[preview_branch_name].head()

Export completed:
- C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\Normal_1_dataset_LCA_B01_Normal_1_reference_values.xlsx
- C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\Normal_1_dataset_LCA_B02_Normal_1_reference_values.xlsx
- C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\Normal_1_dataset_LCA_B03_Normal_1_reference_values.xlsx
- C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\Normal_1_dataset_LCA_B04_Normal_1_reference_values.xlsx
- C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\Normal_1_dataset_LCA_B05_Normal_1_reference_values.xlsx
- C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\Normal_1_dataset_LCA_B06_Normal_1_reference_values.xlsx
- C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\Normal_1_dataset_LCA_B07_Normal_1_reference_values.xlsx
- C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\Normal_1_dataset_RCA_B01_Normal_1_reference_values.xlsx
- C:\Users\adria\OneDr

,Sample_ID,Artery_Type,Branch_ID,Path_Point_Index,Px,Py,Pz,Radius,PointType,Area,gd,Area_prox,Area_dist,A_ref
0,1,LCA,LCA_B01,0,221.379944,242.227921,-91.906563,2.074208,Ostium,24.449387,0.000000,NaN,21.951380,NaN
1,1,LCA,LCA_B01,1,221.518311,242.255295,-91.899139,2.157079,Standard,23.094696,0.141244,NaN,24.636494,NaN
2,1,LCA,LCA_B01,2,221.518860,242.255402,-91.899109,2.157411,Standard,23.094696,0.141804,NaN,24.636494,NaN
3,1,LCA,LCA_B01,3,221.521317,242.257904,-91.896332,2.156677,Standard,23.094696,0.146277,NaN,24.636494,NaN
4,1,LCA,LCA_B01,4,221.791122,242.526337,-91.656708,2.104106,Standard,16.225379,0.596023,NaN,27.313295,NaN
